In [ ]:
import numpy as np
import pickle
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [ ]:
X_text = np.load("../models/X_text.npy")       # tokenized + padded text
y_genre = np.load("../models/y_genre.npy")     # encoded genre labels

with open("../models/text_tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

print(X_text.shape, y_genre.shape)


In [ ]:
vocab_size = len(tokenizer.word_index) + 1
max_length = X_text.shape[1]
num_genres = len(np.unique(y_genre))

print("Vocab size:", vocab_size)
print("Max length:", max_length)
print("Number of genres:", num_genres)


In [ ]:
text_model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128, input_length=max_length),
    LSTM(128, return_sequences=False),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dense(num_genres, activation="softmax")
])

text_model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

text_model.summary()


In [ ]:
history = text_model.fit(
    X_text,
    y_genre,
    epochs=15,
    batch_size=32,
    validation_split=0.2
)


In [ ]:
text_model.save("../models/text_lstm_genre_model.h5")
print("Text LSTM model saved")


In [ ]:
sample_text = ["a calm classical piano piece with slow tempo"]

seq = tokenizer.texts_to_sequences(sample_text)
padded = pad_sequences(seq, maxlen=max_length)

prediction = text_model.predict(padded)
predicted_genre = np.argmax(prediction)

print("Predicted genre index:", predicted_genre)
